In [3]:
!pip install transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 7.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=dca5fe5f15b74308f92b3d13a8ed272ae5be899c38bcfbeb9838e88f10609e43
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [4]:
"""
gpt2_substrate_benchmark_agent.py
=================================
Milestone 1 Benchmark Agent on GPT-2 Small Substrate.
Features:
1. Substrate-Wide Scan: 144 Attention Heads + 12 MLP Layers + Residual Streams.
2. Hoisted Cache Architecture: Computes baseline/clean cache once per task,
   reducing redundant forward passes from 432 to 3.
3. Expanded SelfGraph Knowledge Unit: Tracks Causal Delta, Variance, and Confidence.
4. Autonomous Regime Discovery & Predictive Re-identification.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set
from transformer_lens import HookedTransformer


# =============================================================================
# 1. GPT-2 Substrate Environment with Hoisted Caching
# =============================================================================

def make_head_patch_hook(clean_cache, layer_idx: int, head_idx: int):
    def hook_fn(act, hook):
        act[:, -1, head_idx, :] = clean_cache[f"blocks.{layer_idx}.attn.hook_z"][:, -1, head_idx, :]
        return act
    return hook_fn

def make_mlp_patch_hook(clean_cache, layer_idx: int):
    def hook_fn(act, hook):
        act[:, -1, :] = clean_cache[f"blocks.{layer_idx}.hook_mlp_out"][:, -1, :]
        return act
    return hook_fn


class SubstrateGPT2Env:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.n_layers = self.model.cfg.n_layers
        self.n_heads = self.model.cfg.n_heads

        self.all_144_heads = [(l, h) for l in range(self.n_layers) for h in range(self.n_heads)]
        self.all_mlp_layers = list(range(self.n_layers))

    def get_task_baseline_and_cache(self, task: Dict) -> Tuple[float, torch.Tensor, Dict]:
        clean_toks = self.model.to_tokens(task["clean"])
        corrupted_toks = self.model.to_tokens(task["corrupted"])

        assert clean_toks.shape[1] == corrupted_toks.shape[1], \
            f"Token mismatch: Clean ({clean_toks.shape[1]}) vs Corrupted ({corrupted_toks.shape[1]})"

        correct_id = int(self.model.to_single_token(task["correct"]))
        incorrect_id = int(self.model.to_single_token(task["incorrect"]))

        with torch.no_grad():
            _, clean_cache = self.model.run_with_cache(clean_toks)
            base_logits = self.model(corrupted_toks)

        base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
        return base_diff, corrupted_toks, clean_cache

    def execute_patched_head_fast(
        self, corrupted_toks: torch.Tensor, clean_cache: Dict, task: Dict, head_tuples: List[Tuple[int, int]]
    ) -> float:
        correct_id = int(self.model.to_single_token(task["correct"]))
        incorrect_id = int(self.model.to_single_token(task["incorrect"]))

        hooks = [(f"blocks.{l}.attn.hook_z", make_head_patch_hook(clean_cache, l, h)) for l, h in head_tuples]

        with torch.no_grad():
            logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

        return float((logits[0, -1, correct_id] - logits[0, -1, incorrect_id]).item())

    def execute_patched_mlp_fast(
        self, corrupted_toks: torch.Tensor, clean_cache: Dict, task: Dict, layer_idx: int
    ) -> float:
        correct_id = int(self.model.to_single_token(task["correct"]))
        incorrect_id = int(self.model.to_single_token(task["incorrect"]))

        hooks = [(f"blocks.{layer_idx}.hook_mlp_out", make_mlp_patch_hook(clean_cache, layer_idx))]

        with torch.no_grad():
            logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

        return float((logits[0, -1, correct_id] - logits[0, -1, incorrect_id]).item())


# =============================================================================
# 2. Advanced Causal Knowledge Unit & SelfGraph
# =============================================================================

@dataclass
class CausalKnowledgeUnit:
    component_id: str
    effect_map: Dict[str, float] = field(default_factory=dict)
    variance_map: Dict[str, float] = field(default_factory=dict)

    def record(self, regime_id: str, mean_effect: float, var_effect: float, noise_floor: float = 0.10):
        if abs(mean_effect) < noise_floor:
            mean_effect = 0.0
            var_effect = 0.0
        self.effect_map[regime_id] = mean_effect
        self.variance_map[regime_id] = var_effect

    def get_effect(self, regime_id: str) -> float:
        return self.effect_map.get(regime_id, 0.0)


class BenchmarkSelfGraph:
    def __init__(self):
        self.head_effects: Dict[Tuple[int, int], CausalKnowledgeUnit] = {}
        self.mlp_effects: Dict[int, CausalKnowledgeUnit] = {}
        self.synergies: Dict[Tuple[Tuple[int, int], Tuple[int, int]], CausalKnowledgeUnit] = {}
        self.active_regime_id: str = "R_UNINITIALIZED"

    def predict_delta(self, patched_heads: List[Tuple[int, int]]) -> float:
        r_id = self.active_regime_id
        pred_delta = 0.0
        for h in patched_heads:
            if h in self.head_effects:
                pred_delta += self.head_effects[h].get_effect(r_id)

        for i in range(len(patched_heads)):
            for j in range(i + 1, len(patched_heads)):
                pair = (patched_heads[i], patched_heads[j])
                pair_key = pair if pair[0] < pair[1] else (pair[1], pair[0])
                if pair_key in self.synergies:
                    pred_delta += self.synergies[pair_key].get_effect(r_id)

        return pred_delta


# =============================================================================
# 3. Substrate Benchmark Cognitive Agent
# =============================================================================

class SubstrateBenchmarkAgent:
    def __init__(self, env: SubstrateGPT2Env, base_error_threshold: float = 0.35):
        self.env = env
        self.self_graph = BenchmarkSelfGraph()
        self.known_regimes: List[str] = []
        self.error_threshold = base_error_threshold
        self.adaptive_probing_heads: Set[Tuple[int, int]] = set()

    def get_current_probing_set(self) -> List[Tuple[int, int]]:
        if not self.adaptive_probing_heads:
            return [(7, 4), (8, 10)]
        return list(self.adaptive_probing_heads)

    def evaluate_predictive_mse(self, tasks: List[Dict], regime_id: str) -> float:
        self.self_graph.active_regime_id = regime_id
        probing_set = self.get_current_probing_set()
        errors = []

        # Hoisting baseline and cache for MSE evaluation
        task_caches = [self.env.get_task_baseline_and_cache(t) for t in tasks]

        for idx, task in enumerate(tasks):
            base_diff, corrupted_toks, clean_cache = task_caches[idx]
            for h in probing_set:
                patched_diff = self.env.execute_patched_head_fast(corrupted_toks, clean_cache, task, [h])
                actual_delta = patched_diff - base_diff
                pred_delta = self.self_graph.predict_delta([h])
                errors.append((actual_delta - pred_delta) ** 2)

        return float(np.mean(errors))

    def detect_and_switch_regime(self, sample_tasks: List[Dict]) -> Tuple[str, bool]:
        best_regime = None
        best_mse = float('inf')

        for r_id in self.known_regimes:
            mse = self.evaluate_predictive_mse(sample_tasks, r_id)
            if mse < best_mse:
                best_mse = mse
                best_regime = r_id

        if best_mse < self.error_threshold and best_regime is not None:
            self.self_graph.active_regime_id = best_regime
            print(f"  💡 [RE-IDENTIFICATION] Matched existing LLM Regime: '{best_regime}' (MSE: {best_mse:.4f})")
            return (best_regime, False)
        else:
            new_r_id = f"R_{len(self.known_regimes)}"
            self.known_regimes.append(new_r_id)
            self.self_graph.active_regime_id = new_r_id
            print(f"  🚨 [CAUSAL INCONSISTENCY] Predictive MSE ({best_mse:.4f}) > Threshold ({self.error_threshold})")
            print(f"  🌟 [NEW REGIME CREATED] Internal label assigned: '{new_r_id}'")
            return (new_r_id, True)

    def execute_full_substrate_discovery(self, discovery_tasks: List[Dict], top_k_synergy: int = 4):
        r_id = self.self_graph.active_regime_id
        print(f"  🔬 [STAGE 1: Substrate Scan] Scanning 144 Heads + 12 MLPs for Regime '{r_id}'...")

        # HOISTING CACHE: Calculate clean cache once for all discovery tasks
        task_caches = [self.env.get_task_baseline_and_cache(t) for t in discovery_tasks]

        head_deltas_map: Dict[Tuple[int, int], float] = {}

        # 1. Scan 144 Attention Heads
        for l, h in self.env.all_144_heads:
            deltas = []
            for idx, task in enumerate(discovery_tasks):
                base_diff, corrupted_toks, clean_cache = task_caches[idx]
                patched_diff = self.env.execute_patched_head_fast(corrupted_toks, clean_cache, task, [(l, h)])
                deltas.append(patched_diff - base_diff)

            mean_delta = float(np.mean(deltas))
            var_delta = float(np.var(deltas))
            head_key = (l, h)
            head_deltas_map[head_key] = mean_delta

            if head_key not in self.self_graph.head_effects:
                self.self_graph.head_effects[head_key] = CausalKnowledgeUnit(f"L{l}H{h}")
            self.self_graph.head_effects[head_key].record(r_id, mean_delta, var_delta, noise_floor=0.15)

            if abs(mean_delta) >= 0.20:
                self.adaptive_probing_heads.add(head_key)

        # 2. Scan 12 MLP Layers
        for l in self.env.all_mlp_layers:
            deltas = []
            for idx, task in enumerate(discovery_tasks):
                base_diff, corrupted_toks, clean_cache = task_caches[idx]
                patched_diff = self.env.execute_patched_mlp_fast(corrupted_toks, clean_cache, task, l)
                deltas.append(patched_diff - base_diff)

            mean_delta = float(np.mean(deltas))
            var_delta = float(np.var(deltas))
            if l not in self.self_graph.mlp_effects:
                self.self_graph.mlp_effects[l] = CausalKnowledgeUnit(f"MLP_{l}")
            self.self_graph.mlp_effects[l].record(r_id, mean_delta, var_delta, noise_floor=0.15)

        # 3. Fine Synergy Scan on Top Heads
        sorted_discovered_heads = sorted(head_deltas_map.items(), key=lambda x: abs(x[1]), reverse=True)
        top_heads = [h_tuple for h_tuple, val in sorted_discovered_heads[:top_k_synergy] if abs(val) >= 0.20]

        print(f"  🎯 [STAGE 2: Fine Synergy Scan] Top {len(top_heads)} Active Heads for Regime '{r_id}': {top_heads}")

        for i in range(len(top_heads)):
            for j in range(i + 1, len(top_heads)):
                h1, h2 = top_heads[i], top_heads[j]
                synergies = []

                for idx, task in enumerate(discovery_tasks):
                    base_diff, corrupted_toks, clean_cache = task_caches[idx]
                    d1 = self.env.execute_patched_head_fast(corrupted_toks, clean_cache, task, [h1]) - base_diff
                    d2 = self.env.execute_patched_head_fast(corrupted_toks, clean_cache, task, [h2]) - base_diff
                    d12 = self.env.execute_patched_head_fast(corrupted_toks, clean_cache, task, [h1, h2]) - base_diff

                    syn = d12 - (d1 + d2)
                    synergies.append(syn)

                pair_key = (h1, h2) if h1 < h2 else (h2, h1)
                if pair_key not in self.self_graph.synergies:
                    self.self_graph.synergies[pair_key] = CausalKnowledgeUnit(f"Syn_{h1}_{h2}")
                self.self_graph.synergies[pair_key].record(r_id, float(np.mean(synergies)), float(np.var(synergies)), noise_floor=0.10)

        self._print_selfgraph_status()

    def _print_selfgraph_status(self):
        r_id = self.self_graph.active_regime_id
        print("\n  ==================================================")
        print(f"  🧠 SUBSTRATE SELFGRAPH DISCOVERED FOR '{r_id}':")
        print("  ==================================================")

        print("  [Active Attention Heads]")
        active_heads = [(h, ku.get_effect(r_id)) for h, ku in self.self_graph.head_effects.items() if abs(ku.get_effect(r_id)) > 0.15]
        active_heads.sort(key=lambda x: abs(x[1]), reverse=True)
        for h, eff in active_heads:
            print(f"    ★ Head L{h[0]}H{h[1]:<2} ──► Delta: {eff:+6.3f}")

        print("  [Active MLP Layers]")
        active_mlps = [(l, ku.get_effect(r_id)) for l, ku in self.self_graph.mlp_effects.items() if abs(ku.get_effect(r_id)) > 0.15]
        active_mlps.sort(key=lambda x: abs(x[1]), reverse=True)
        for l, eff in active_mlps:
            print(f"    ⚙️ MLP Layer {l:<2} ──► Delta: {eff:+6.3f}")

        for pair, ku in self.self_graph.synergies.items():
            syn = ku.get_effect(r_id)
            if abs(syn) > 0.10:
                print(f"    ⚡️ Synergy (L{pair[0][0]}H{pair[0][1]}, L{pair[1][0]}H{pair[1][1]}) ──► {syn:+6.3f}")
        print("  ==================================================\n")


# =============================================================================
# 4. Pipeline Execution
# =============================================================================

if __name__ == "__main__":
    print("==================================================================")
    print("🚀 SUBSTRATE BENCHMARK AGENT ON GPT-2 SMALL")
    print("==================================================================")

    env = SubstrateGPT2Env("gpt2")
    agent = SubstrateBenchmarkAgent(env, base_error_threshold=0.35)

    SVA_TASKS = [
        {"clean": "The key to the cabinets", "corrupted": "The keys to the cabinets", "correct": " is", "incorrect": " are"},
        {"clean": "The book on the tables", "corrupted": "The books on the tables", "correct": " is", "incorrect": " are"},
        {"clean": "The door to the rooms", "corrupted": "The doors to the rooms", "correct": " is", "incorrect": " are"}
    ]

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"},
        {"clean": "Then, David and Sarah went to the office. David gave a letter to", "corrupted": "Then, David and Sarah went to the office. Sarah gave a letter to", "correct": " Sarah", "incorrect": " David"}
    ]

    regime_task_sequence = [
        ("SVA Task 1", SVA_TASKS),
        ("SVA Task 2", SVA_TASKS),
        ("IOI Task 1", IOI_TASKS),
        ("IOI Task 2", IOI_TASKS),
        ("SVA Task 3 (Return)", SVA_TASKS),
    ]

    for step, (task_name, task_dataset) in enumerate(regime_task_sequence, 1):
        print(f"\n------------------------------------------------------------------")
        print(f"📌 [STEP {step}/5] Input Task Sequence: '{task_name}'")
        print(f"------------------------------------------------------------------")

        active_r_id, is_new = agent.detect_and_switch_regime(task_dataset)

        if is_new:
            agent.execute_full_substrate_discovery(task_dataset, top_k_synergy=4)

        post_mse = agent.evaluate_predictive_mse(task_dataset, active_r_id)
        print(f"  🎯 Active State: '{agent.self_graph.active_regime_id}' | Post-Prediction MSE: {post_mse:.4f}")

🚀 SUBSTRATE BENCHMARK AGENT ON GPT-2 SMALL


/tmp/ipykernel_1054/2220584642.py:40: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer

------------------------------------------------------------------
📌 [STEP 1/5] Input Task Sequence: 'SVA Task 1'
------------------------------------------------------------------
  🚨 [CAUSAL INCONSISTENCY] Predictive MSE (inf) > Threshold (0.35)
  🌟 [NEW REGIME CREATED] Internal label assigned: 'R_0'
  🔬 [STAGE 1: Substrate Scan] Scanning 144 Heads + 12 MLPs for Regime 'R_0'...
  🎯 [STAGE 2: Fine Synergy Scan] Top 4 Active Heads for Regime 'R_0': [(7, 4), (10, 9), (8, 5), (10, 5)]

  🧠 SUBSTRATE SELFGRAPH DISCOVERED FOR 'R_0':
  [Active Attention Heads]
    ★ Head L7H4  ──► Delta: +2.083
    ★ Head L10H9  ──► Delta: +1.475
    ★ Head L8H5  ──► Delta: +1.449
    ★ Head L10H5  ──► Delta: +0.484
    ★ Head L6H0  ──► Delta: +0.426
    ★ Head L11H10 ──► Delta: +0.409
    ★ Head L5H2  ──► Delta: +0.187
  [Active MLP Layers]
    ⚙️ MLP Layer 10 ──► Delta: +1.517
    ⚙️ MLP Layer 11 ──► Delta: +1.223
    ⚙️ MLP Layer 9  ──► Delta: +1.186
 